In [ ]:
# 이 칸은 그대로 실행하세요. 데이터 경로와 그래프 한글 폰트를 잡아둡니다.
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager, rcParams

DATA_DIR = os.path.join("..", "data")          # notebooks 폴더 기준 한 단계 위의 data 폴더
FIG_DIR = os.path.join("..", "figures")        # 그래프 저장 위치
os.makedirs(FIG_DIR, exist_ok=True)

# 기존 3개 (T- 센서 계측, G-02 사건 기록, G-01 기준 정보)
TEMP_CSV = os.path.join(DATA_DIR, "T-CR1-CAL01_온도.csv")          # 소둔로 온도·제어출력, 1초 주기
EVENT_CSV = os.path.join(DATA_DIR, "G-02_조업이벤트.csv")           # 강종변경 등, PLANT_CD 단위
REPAIR_CSV = os.path.join(DATA_DIR, "G-01_정기수리캘린더.csv")       # 계획 정지, PLANT_CD 단위

# 2026-09-28 추가된 4개 (상위 이슈 #43)
EQP_CSV = os.path.join(DATA_DIR, "G-01_설비마스터.csv")             # 설비 사전, 키 EQP_TAG
PART_CSV = os.path.join(DATA_DIR, "G-01_부품마스터.csv")            # 부품 사전, 키 PART_CD
MAINT_CSV = os.path.join(DATA_DIR, "G-02_정비이력.csv")             # 정비·고장 기록, 키 WO_NO + EQP_TAG
PARTCHG_CSV = os.path.join(DATA_DIR, "G-02_부품교체이력.csv")        # 부품 교체, 키 WO_NO + PART_CD

ENC = "utf-8-sig"                              # G- 로 시작하는 마스터 CSV는 BOM 있음

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

# 그래프 한글 깨짐 방지 - 설치된 폰트 중 있는 것을 골라 rcParams에 넣습니다
installed = {f.name for f in font_manager.fontManager.ttflist}
for cand in ["Malgun Gothic", "AppleGothic", "NanumGothic", "NanumBarunGothic", "Noto Sans CJK KR"]:
    if cand in installed:
        rcParams["font.family"] = cand
        break
else:
    print("경고: 한글 폰트를 못 찾았습니다. 그래프의 한글이 □ 로 보일 수 있습니다.")

rcParams["axes.unicode_minus"] = False         # 음수 기호가 □ 로 나오는 것 방지
rcParams["figure.dpi"] = 110

---
# `G-02_정비이력.csv` 분석

In [ ]:
# G-02_정비이력.csv 데이터 로드 및 결과 확인
import pathlib

# 상단 초기화 셀에 정의된 경로(MAINT_CSV)와 인코딩(ENC)을 사용하여 데이터 로드
df = pd.read_csv(MAINT_CSV, encoding=ENC)
file_path = pathlib.Path(MAINT_CSV)

print(f"✅ 데이터 로드 완료: {file_path.name}")
print(f"전체 규모: {df.shape[0]:,}행 × {df.shape[1]}열\n")

# `G-02_정비이력.csv`의 컬럼과 결측치 파악

In [ ]:
# 컬럼별 주요 상태 요약 표
col_summary = pd.DataFrame({
    "컬럼명": df.columns,
    "데이터 타입": df.dtypes.values,
    "결측치 수": df.isnull().sum().values,
    "유효 데이터 수": df.notnull().sum().values
})

display(col_summary)

## 📊 `G-02_정비이력.csv` 컬럼별 의미와 결측치 요약표

| 번호 | 컬럼명 | 한글명 | 결측치 수 | 유효 데이터 수 |
| :---: | :--- | :--- | :---: | :---: |
| 1 | `WO_NO` | 작업지시번호 | 0 | 2,112 |
| 2 | `EQP_TAG` | 설비태그 | 0 | 2,112 |
| 3 | `DETC_DT` | 고장/이상 감지일시 | 0 | 2,112 |
| 4 | `STRT_DT` | 정비 시작일시 | 0 | 2,112 |
| 5 | `END_DT` | 정비 종료일시 | 0 | 2,112 |
| 6 | `MNT_TYPE` | 정비유형 | 0 | 2,112 |
| 7 | `FAIL_MODE` | 고장모드 | 1,221 | 891 |
| 8 | `DOWN_MIN` | 다운타임(분) | 0 | 2,112 |
| 9 | `PART_CHG` | 부품교체 여부/상세 | 0 | 2,112 |
| 10 | `WRKR_ID` | 작업자 ID | 0 | 2,112 |
| 11 | `WRK_TEAM` | 작업 팀 | 0 | 2,112 |
| 12 | `REMARK` | 비고/작업내용 | 314 | 1,798 |

---
## 🔍 주요 컬럼 간 관계 분석 및 심층 탐색
본 섹션에서는 정비이력 데이터에서 도출할 수 있는 주요 변수 간의 상관관계와 패턴을 다각도로 분석합니다.

* **1. 정비유형 (`MNT_TYPE`) ↔ 다운타임 (`DOWN_MIN`)**: 예방정비와 사후정비에 따른 설비 정지 시간 비교
* **2. 설비태그 (`EQP_TAG`) ↔ 고장모드 (`FAIL_MODE`)**: 설비별 주요 고장 유형 집중 분석
* **3. 정비 리드타임 및 대기시간 분석 (`DETC_DT` ↔ `STRT_DT` ↔ `END_DT`)**: 이상 감지부터 정비 완료까지의 프로세스 효율성 진단
* **4. 작업 팀별 정비 효율성 분석 (`WRK_TEAM` ↔ `DOWN_MIN`)**: 조직별 정비 소요 시간 및 대응 속도 비교
* **5. 부품 교체 이력 연계 분석 (`PARTCHG_CSV`)**: 부품 교체가 설비 정지 시간에 미치는 영향 파악
* **6. 조업 이벤트 연계 분석 (`EVENT_CSV`)**: 조업 변동 전후 고장 발생 패턴 파악

### 1. 정비유형별 평균 다운타임 분석
* **분석 목적**: 예방정비(PM)와 사후정비(BM) 중 어떤 유형이 설비 가동 중단 시간(다운타임)에 더 큰 영향을 주는지 확인합니다.

In [ ]:
# 정비유형에 따른 다운타임 평균 및 총합 계산
mnt_down_df = df.groupby('MNT_TYPE')['DOWN_MIN'].agg(['count', 'mean', 'sum']).reset_index()
mnt_down_df.columns = ['정비유형', '정비건수', '평균다운타임(분)', '총다운타임(분)']

# 보기 쉽게 반올림
display(mnt_down_df.round(1))

📌 **[분석 결과 요약 및 인사이트]**
* **사후정비(BM)** 및 **상태기반보전(CBM)**의 평균 다운타임은 약 **275~276분**으로, 정기예방정비인 **TBM(113.2분)**에 비해 설비 정지 시간이 **2배 이상 길게** 나타났습니다.
* 따라서 돌발 고장으로 이어지는 BM/CBM 발생 빈도를 줄이는 사전 예방 체계 강화가 필요합니다.
---

### 2. 설비태그별 주요 고장모드 분석
* **분석 목적**: 특정 설비(`EQP_TAG`)에서 자주 발생하는 고장모드(`FAIL_MODE`)의 패턴을 파악하여 취약 요소를 도출합니다.

In [ ]:
# 설비태그와 고장모드 교차표(Crosstab) 생성
fail_cross = pd.crosstab(df['EQP_TAG'], df['FAIL_MODE'], margins=True)
display(fail_cross)

📌 **[분석 결과 요약 및 인사이트]**
* 전체 891건의 고장모드 중 **마모(WER)**와 **전기적 원인(ELE/ETC)** 관련 고장이 빈번하게 발생하고 있습니다.
* 특정 설비 태그에 고장이 집중되는 경향이 있으므로 해당 설비에 대한 정밀 점검 주기 단축을 검토해야 합니다.
---

### 3. 정비 프로세스별 리드타임 및 대기시간 분석
* **분석 목적**: 이상 감지(`DETC_DT`)부터 실제 정비 시작(`STRT_DT`)까지의 대기 시간과, 순수 정비 소요 시간(`STRT_DT` ~ `END_DT`)을 분리하여 현장 대응력을 진단합니다.

In [ ]:
# 날짜/시간 컬럼을 datetime 형식으로 변환
for col in ['DETC_DT', 'STRT_DT', 'END_DT']:
    df[col] = pd.to_datetime(df[col])

# 대기시간 (감지 ~ 정비시작) 과 순수 정비시간 (정비시작 ~ 종료) 계산 (분 단위)
df['WAIT_MIN'] = (df['STRT_DT'] - df['DETC_DT']).dt.total_seconds() / 60
df['PURE_MNT_MIN'] = (df['END_DT'] - df['STRT_DT']).dt.total_seconds() / 60

# 정비유형별 평균 대기시간 및 순수 정비시간 요약
time_analysis = df.groupby('MNT_TYPE')[['WAIT_MIN', 'PURE_MNT_MIN', 'DOWN_MIN']].mean().reset_index()
time_analysis.columns = ['정비유형', '평균 대기시간(분)', '평균 순수정비시간(분)', '평균 총다운타임(분)']

display(time_analysis.round(1))

📌 **[분석 결과 요약 및 인사이트]**
* 사후정비(BM)의 경우 고장 감지 후 정비 착수까지의 대기 시간과 실제 수리 시간이 모두 길게 나타나는 경향이 있습니다.
* 신속한 초동 조치를 통해 대기 시간을 단축하는 것이 전체 다운타임 축소의 핵심 요인입니다.
---

### 4. 작업 팀별 정비 효율성 및 다운타임 분석
* **분석 목적**: 작업 팀(`WRK_TEAM`)별로 정비 건수와 평균 다운타임을 비교하여 팀별 작업 특성과 숙련도를 파악합니다.

In [ ]:
# 작업 팀별 정비 건수 및 평균 다운타임 계산
team_analysis = df.groupby('WRK_TEAM')['DOWN_MIN'].agg(['count', 'mean', 'sum']).reset_index()
team_analysis.columns = ['작업팀', '정비건수', '평균다운타임(분)', '총다운타임(분)']

display(team_analysis.round(1))

📌 **[분석 결과 요약 및 인사이트]**
* 작업 팀별로 평균 정지 시간에 차이가 발생하는지 확인하고, 정비 표준화가 필요한 조직을 식별할 수 있습니다.
---

### 5. 부품 교체 이력 데이터 연계 분석
* **분석 목적**: `G-02_정비이력.csv`와 부품 교체 이력(`PARTCHG_CSV`) 데이터를 작업지시번호(`WO_NO`) 기준으로 병합하여 부품 교체가 다운타임에 미치는 영향을 분석합니다.

In [ ]:
# 부품 교체 이력 데이터 로드
df_partchg = pd.read_csv(PARTCHG_CSV, encoding=ENC)

# WO_NO 기준으로 정비이력과 부품교체이력 병합 (Inner 혹은 Left Join)
df_merged = pd.merge(df, df_partchg, on='WO_NO', how='left')

print(f"병합된 데이터 규모: {df_merged.shape[0]:,}행 × {df_merged.shape[1]}열")
display(df_merged.head(3))

📌 **[분석 결과 요약 및 인사이트]**
* 부품 교체가 수반된 작업과 단순 조정 작업 간의 다운타임 차이를 비교하여 핵심 소모성 부품 관리를 최적화할 수 있습니다.
---

### 6. 조업 이벤트 데이터 연계 분석
* **분석 목적**: 강종 변경 등 조업 이벤트(`EVENT_CSV`) 발생 시점과 정비 이력의 고장 감지 시점(`DETC_DT`)을 연계하여 조업 변화가 설비 고장에 미치는 인과관계를 살펴봅니다.

In [ ]:
# 조업 이벤트 데이터 로드 확인
df_event = pd.read_csv(EVENT_CSV, encoding=ENC)
print(f"조업 이벤트 데이터 규모: {df_event.shape[0]:,}행 × {df_event.shape[1]}열")
display(df_event.head(3))

📌 **[분석 결과 요약 및 인사이트]**
* 특정 조업 이벤트(예: 고부하 강종 전환) 직후에 돌발 고장(BM)이 집중되는지 파악하여 사전 모니터링 체계를 강화할 수 있습니다.
---

---
## 🔍 개별 컬럼 상세 탐색 (단변량 분석 및 데이터 상태 검증)

---
### <span id="desc-wo-no"></span>1. `WO_NO` (작업지시번호)
- **약어 풀이**: **W**ork **O**rder **N**umber
- **의미**: 각 정비/수리 작업마다 부여되는 고유 작업 지시 번호
- **데이터 상태**: 결측치 0건 (완전한 데이터)

In [ ]:
col = 'WO_NO'

# 1. 결측치 개수 및 기본 통계 계산
total_cnt = len(df)
null_cnt = df[col].isnull().sum()
null_rate = (null_cnt / total_cnt) * 100
unique_cnt = df[col].nunique(dropna=True)

print(f"=== [{col}] 작업지시번호 데이터 현황 ===")
print(f"· 전체 데이터 건수 : {total_cnt:,}건")
print(f"· 고유 ID 개수(Unique) : {unique_cnt:,}개")
print(f"· 결측치(Null) 건수 : {null_cnt:,}건 ({null_rate:.2f}%)")

# 2. 결측 행 존재 시 데이터 확인
if null_cnt > 0:
    print(f"\n⚠️ [{col}] 결측 데이터 샘플 (상위 5건):")
    display(df[df[col].isnull()].head())
else:
    print(f"\n✅ [{col}] 결측치가 존재하지 않습니다.")

---
### <span id="desc-eqp-tag"></span>2. `EQP_TAG` (설비태그)
- **약어 풀이**: **Eq**uip**m**ent **Tag**
- **의미**: 정비가 수행된 대상 설비 식별 코드 (예: CAL01 등)
- **데이터 상태**: 결측치 0건

In [ ]:
col = 'EQP_TAG'

# 1. 결측치 개수 및 기본 통계 계산
total_cnt = len(df)
null_cnt = df[col].isnull().sum()
null_rate = (null_cnt / total_cnt) * 100
unique_cnt = df[col].nunique(dropna=True)

print(f"=== [{col}] 설비태그 데이터 현황 ===")
print(f"· 전체 데이터 건수 : {total_cnt:,}건")
print(f"· 고유 태그 개수(Unique) : {unique_cnt:,}개")
print(f"· 결측치(Null) 건수 : {null_cnt:,}건 ({null_rate:.2f}%)")

# 2. 결측 행 존재 시 데이터 확인
if null_cnt > 0:
    print(f"\n⚠️ [{col}] 결측 데이터 샘플 (상위 5건):")
    display(df[df[col].isnull()].head())
else:
    print(f"\n✅ [{col}] 결측치가 존재하지 않습니다.")

---
### <span id="desc-detc-dt"></span>3. `DETC_DT` (고장/이상 감지일시)
- **약어 풀이**: **Det**e**c**tion **D**a**t**e
- **의미**: 이상 징후 또는 고장이 처음 감지/발견된 계측 시각
- **데이터 상태**: 결측치 0건

In [ ]:
col = 'DETC_DT'

# 1. 결측치 개수 및 기본 통계 계산
total_cnt = len(df)
null_cnt = df[col].isnull().sum()
null_rate = (null_cnt / total_cnt) * 100
unique_cnt = df[col].nunique(dropna=True)

print(f"=== [{col}] 고장/이상 감지일시 데이터 현황 ===")
print(f"· 전체 데이터 건수 : {total_cnt:,}건")
print(f"· 고유 일시 개수(Unique) : {unique_cnt:,}개")
print(f"· 결측치(Null) 건수 : {null_cnt:,}건 ({null_rate:.2f}%)")

# 2. 결측 행 존재 시 데이터 확인
if null_cnt > 0:
    print(f"\n⚠️ [{col}] 결측 데이터 샘플 (상위 5건):")
    display(df[df[col].isnull()].head())
else:
    print(f"\n✅ [{col}] 결측치가 존재하지 않습니다.")

---
### <span id="desc-strt-dt"></span>4. `STRT_DT` (정비 시작일시)
- **약어 풀이**: **St**a**rt** **D**a**t**e
- **의미**: 정비 작업이 실제로 시작된 시각
- **데이터 상태**: 결측치 0건

In [ ]:
col = 'STRT_DT'

# 1. 결측치 개수 및 기본 통계 계산
total_cnt = len(df)
null_cnt = df[col].isnull().sum()
null_rate = (null_cnt / total_cnt) * 100
unique_cnt = df[col].nunique(dropna=True)

print(f"=== [{col}] 정비 시작일시 데이터 현황 ===")
print(f"· 전체 데이터 건수 : {total_cnt:,}건")
print(f"· 고유 일시 개수(Unique) : {unique_cnt:,}개")
print(f"· 결측치(Null) 건수 : {null_cnt:,}건 ({null_rate:.2f}%)")

# 2. 결측 행 존재 시 데이터 확인
if null_cnt > 0:
    print(f"\n⚠️ [{col}] 결측 데이터 샘플 (상위 5건):")
    display(df[df[col].isnull()].head())
else:
    print(f"\n✅ [{col}] 결측치가 존재하지 않습니다.")

---
### <span id="desc-end-dt"></span>5. `END_DT` (정비 종료일시)
- **약어 풀이**: **E**n**d** **D**a**t**e
- **의미**: 정비 작업이 완료되어 설비가 정상 가동된 시각
- **데이터 상태**: 결측치 0건

In [ ]:
col = 'END_DT'

# 1. 결측치 개수 및 기본 통계 계산
total_cnt = len(df)
null_cnt = df[col].isnull().sum()
null_rate = (null_cnt / total_cnt) * 100
unique_cnt = df[col].nunique(dropna=True)

print(f"=== [{col}] 정비 종료일시 데이터 현황 ===")
print(f"· 전체 데이터 건수 : {total_cnt:,}건")
print(f"· 고유 일시 개수(Unique) : {unique_cnt:,}개")
print(f"· 결측치(Null) 건수 : {null_cnt:,}건 ({null_rate:.2f}%)")

# 2. 결측 행 존재 시 데이터 확인
if null_cnt > 0:
    print(f"\n⚠️ [{col}] 결측 데이터 샘플 (상위 5건):")
    display(df[df[col].isnull()].head())
else:
    print(f"\n✅ [{col}] 결측치가 존재하지 않습니다.")

---
### <span id="desc-mnt-type"></span>6. `MNT_TYPE` (정비유형)
- **약어 풀이**: **M**ai**n**te**n**ance **Type**
- **의미**: 정비 목적 및 방식 분류 (`BM`: 사후보전, `CBM`: 상태기반보전, `TBM`: 시간기반보전 등)
- **데이터 상태**: 결측치 0건

In [ ]:
col = 'MNT_TYPE'

# 1. 결측치 개수 및 기본 통계 계산
total_cnt = len(df)
null_cnt = df[col].isnull().sum()
null_rate = (null_cnt / total_cnt) * 100
unique_cnt = df[col].nunique(dropna=True)

print(f"=== [{col}] 정비유형 데이터 현황 ===")
print(f"· 전체 데이터 건수 : {total_cnt:,}건")
print(f"· 고유 유형 개수(Unique) : {unique_cnt:,}개")
print(f"· 결측치(Null) 건수 : {null_cnt:,}건 ({null_rate:.2f}%)")

# 2. 결측 행 존재 시 데이터 확인
if null_cnt > 0:
    print(f"\n⚠️ [{col}] 결측 데이터 샘플 (상위 5건):")
    display(df[df[col].isnull()].head())
else:
    print(f"\n✅ [{col}] 결측치가 존재하지 않습니다.")

---
### <span id="desc-fail-mode"></span>7. `FAIL_MODE` (고장모드)
- **약어 풀이**: **Fail**ure **Mode**
- **의미**: 설비 고장 원인 및 현상 분류 (예: 마모, 과열, 누수 등)
- **데이터 상태**: **결측치 1,221건 (약 57.8%)** - 정기점검/예방정비(TBM/CBM)의 경우 고장이 아니므로 결측일 가능성 분석 필요

In [ ]:
col = 'FAIL_MODE'

# 1. 결측치 개수 및 기본 통계 계산
total_cnt = len(df)
null_cnt = df[col].isnull().sum()
null_rate = (null_cnt / total_cnt) * 100
unique_cnt = df[col].nunique(dropna=True)

print(f"=== [{col}] 고장모드 데이터 현황 ===")
print(f"· 전체 데이터 건수 : {total_cnt:,}건")
print(f"· 고유 모드 개수(Unique) : {unique_cnt:,}개")
print(f"· 결측치(Null) 건수 : {null_cnt:,}건 ({null_rate:.2f}%)")

# 2. 결측 행 존재 시 데이터 확인
if null_cnt > 0:
    print(f"\n⚠️ [{col}] 결측 데이터 샘플 (상위 5건):")
    display(df[df[col].isnull()].head())
else:
    print(f"\n✅ [{col}] 결측치가 존재하지 않습니다.")

📌 **[분석 결과 요약 및 인사이트: FAIL_MODE]**
* **높은 결측치 비중 (57.8%)**: 총 2,112건 중 1,221건에 고장모드(`FAIL_MODE`)가 기재되어 있지 않습니다.
* **도메인 특성 해석**: 이는 사후 보전(BM) 작업 시에만 고장모드가 상세히 기록되는 반면, 정기 점검이나 시간기반 예방정비(TBM) 작업의 경우 설비 고장으로 인한 정지가 아니기 때문에 결측 처리되었을 가능성이 매우 높습니다.
* **활용 방안**: 향후 고장 예측 모델링이나 원인 분석 시, 결측치를 단순 제거하기보다는 정비유형(`MNT_TYPE`)과 연계하여 '예방정비 vs 돌발고장'을 구분하는 핵심 필터 조건으로 활용해야 합니다.

---
### <span id="desc-down-min"></span>8. `DOWN_MIN` (다운타임)
- **약어 풀이**: **Down**time **Min**utes
- **의미**: 정비 및 고장으로 인해 설비가 정지된 시간 (단위: 분)
- **데이터 상태**: 결측치 0건 (숫자형 데이터: `int64`)

In [ ]:
col = 'DOWN_MIN'

# 1. 결측치 개수 및 기본 통계 계산
total_cnt = len(df)
null_cnt = df[col].isnull().sum()
null_rate = (null_cnt / total_cnt) * 100
unique_cnt = df[col].nunique(dropna=True)

print(f"=== [{col}] 다운타임(분) 데이터 현황 ===")
print(f"· 전체 데이터 건수 : {total_cnt:,}건")
print(f"· 고유 값 개수(Unique) : {unique_cnt:,}개")
print(f"· 결측치(Null) 건수 : {null_cnt:,}건 ({null_rate:.2f}%)")

# 2. 결측 행 존재 시 데이터 확인
if null_cnt > 0:
    print(f"\n⚠️ [{col}] 결측 데이터 샘플 (상위 5건):")
    display(df[df[col].isnull()].head())
else:
    print(f"\n✅ [{col}] 결측치가 존재하지 않습니다.")

---
### <span id="desc-part-chg"></span>9. `PART_CHG` (부품교체 여부/상세)
- **약어 풀이**: **Part** **Ch**an**g**e
- **의미**: 정비 과정에서의 부품 교체 유무 또는 교체 내역
- **데이터 상태**: 결측치 0건

In [ ]:
col = 'PART_CHG'

# 1. 결측치 개수 및 기본 통계 계산
total_cnt = len(df)
null_cnt = df[col].isnull().sum()
null_rate = (null_cnt / total_cnt) * 100
unique_cnt = df[col].nunique(dropna=True)

print(f"=== [{col}] 부품교체 상세 데이터 현황 ===")
print(f"· 전체 데이터 건수 : {total_cnt:,}건")
print(f"· 고유 항목 개수(Unique) : {unique_cnt:,}개")
print(f"· 결측치(Null) 건수 : {null_cnt:,}건 ({null_rate:.2f}%)")

# 2. 결측 행 존재 시 데이터 확인
if null_cnt > 0:
    print(f"\n⚠️ [{col}] 결측 데이터 샘플 (상위 5건):")
    display(df[df[col].isnull()].head())
else:
    print(f"\n✅ [{col}] 결측치가 존재하지 않습니다.")

---
### <span id="desc-wrkr-id"></span>10. `WRKR_ID` (작업자 ID)
- **약어 풀이**: **Wor**k**e**r **ID**
- **의미**: 정비 작업을 수행한 담당 작업자 식별 번호
- **데이터 상태**: 결측치 0건

In [ ]:
col = 'WRKR_ID'

# 1. 결측치 개수 및 기본 통계 계산
total_cnt = len(df)
null_cnt = df[col].isnull().sum()
null_rate = (null_cnt / total_cnt) * 100
unique_cnt = df[col].nunique(dropna=True)

print(f"=== [{col}] 작업자 ID 데이터 현황 ===")
print(f"· 전체 데이터 건수 : {total_cnt:,}건")
print(f"· 고유 작업자 수(Unique) : {unique_cnt:,}명")
print(f"· 결측치(Null) 건수 : {null_cnt:,}건 ({null_rate:.2f}%)")

# 2. 결측 행 존재 시 데이터 확인
if null_cnt > 0:
    print(f"\n⚠️ [{col}] 결측 데이터 샘플 (상위 5건):")
    display(df[df[col].isnull()].head())
else:
    print(f"\n✅ [{col}] 결측치가 존재하지 않습니다.")

---
### <span id="desc-wrk-team"></span>11. `WRK_TEAM` (작업 팀)
- **약어 풀이**: **Work** **Team**
- **의미**: 정비 작업을 담당한 보전 팀/조 구분
- **데이터 상태**: 결측치 0건

In [ ]:
col = 'WRK_TEAM'

# 1. 결측치 개수 및 기본 통계 계산
total_cnt = len(df)
null_cnt = df[col].isnull().sum()
null_rate = (null_cnt / total_cnt) * 100
unique_cnt = df[col].nunique(dropna=True)

print(f"=== [{col}] 작업 팀 데이터 현황 ===")
print(f"· 전체 데이터 건수 : {total_cnt:,}건")
print(f"· 고유 팀 수(Unique) : {unique_cnt:,}개")
print(f"· 결측치(Null) 건수 : {null_cnt:,}건 ({null_rate:.2f}%)")

# 2. 결측 행 존재 시 데이터 확인
if null_cnt > 0:
    print(f"\n⚠️ [{col}] 결측 데이터 샘플 (상위 5건):")
    display(df[df[col].isnull()].head())
else:
    print(f"\n✅ [{col}] 결측치가 존재하지 않습니다.")

---
### <span id="desc-remark"></span>12. `REMARK` (비고/작업내용)
- **약어 풀이**: **Remark**
- **의미**: 현장 정비 담당자가 작성한 작업 상세 세부 메모 및 특이사항
- **데이터 상태**: **결측치 314건 (약 14.9%)**

In [ ]:
col = 'REMARK'

# 1. 결측치 개수 및 기본 통계 계산
total_cnt = len(df)
null_cnt = df[col].isnull().sum()
null_rate = (null_cnt / total_cnt) * 100
unique_cnt = df[col].nunique(dropna=True)

print(f"=== [{col}] 비고/작업내용 데이터 현황 ===")
print(f"· 전체 데이터 건수 : {total_cnt:,}건")
print(f"· 고유 비고 수(Unique) : {unique_cnt:,}개")
print(f"· 결측치(Null) 건수 : {null_cnt:,}건 ({null_rate:.2f}%)")

# 2. 결측 행 존재 시 데이터 확인
if null_cnt > 0:
    print(f"\n⚠️ [{col}] 결측 데이터 샘플 (상위 5건):")
    display(df[df[col].isnull()].head())
else:
    print(f"\n✅ [{col}] 결측치가 존재하지 않습니다.")

📌 **[분석 결과 요약 및 인사이트: REMARK]**
* **결측치 및 자유 서식 특성**: 전체 데이터 중 약 14.9%(314건)에서 비고란이 비어 있으며, 유효 데이터는 현장 작업자의 정성적 메모(자유 텍스트)로 구성되어 있습니다.
* **도메인 특성 해석**: 정기적인 단순 점검이나 특이사항이 없는 판정의 경우 비고란이 생략되는 경향이 있으며, 반대로 설비 재발 방지나 특이 정비 건의 경우 상세한 코멘트가 기록되어 있습니다.
* **활용 방안**: 정형 데이터 분석 외에 텍스트 마이닝이나 키워드 필터링(예: '재발', '이상', '교체')을 적용하여 현장의 숨겨진 리스크나 반복 이슈를 정성적으로 보완 분석하는 데 활용할 수 있습니다.